# 01 - Extracción de Datos de Competidores

Busca locales gastronómicos competidores en un radio dado usando Google Places API.  
Output: `data/competidores_YYYYMMDD_HHMMSS.csv`

In [1]:
import googlemaps
import pandas as pd
import os
import time
from dotenv import load_dotenv
from datetime import datetime

load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("GOOGLE_PLACES_API_KEY")
gmaps = googlemaps.Client(key=API_KEY)
print("Cliente inicializado correctamente")

ModuleNotFoundError: No module named 'googlemaps'

## Parámetros de búsqueda

Ajustá las coordenadas a la ubicación de tu negocio o zona de interés.

In [ ]:
MI_UBICACION = (-34.6037, -58.3816)  # Latitud, Longitud — reemplazá con tu ubicación
RADIO_METROS = 1000                  # Radio de búsqueda en metros
TIPO_NEGOCIO = "restaurant"          # restaurant | cafe | bar

## Búsqueda de competidores cercanos

Google Places devuelve hasta 60 resultados en 3 páginas de 20.

In [ ]:
respuesta = gmaps.places_nearby(
    location=MI_UBICACION,
    radius=RADIO_METROS,
    type=TIPO_NEGOCIO
)

lugares = respuesta.get("results", [])

while "next_page_token" in respuesta:
    time.sleep(2)  # La API requiere una pequeña espera entre páginas
    respuesta = gmaps.places_nearby(page_token=respuesta["next_page_token"])
    lugares.extend(respuesta.get("results", []))

print(f"Competidores encontrados: {len(lugares)}")

## Extracción de datos detallados

Para cada lugar obtenemos nombre, rating, reseñas, horarios y ubicación exacta.  
> Nota: cada llamada a `place()` consume cuota de la API.

In [ ]:
CAMPOS = [
    "name", "rating", "user_ratings_total", "formatted_address",
    "geometry", "opening_hours", "price_level", "types"
]

datos = []
for lugar in lugares:
    place_id = lugar["place_id"]
    detalles = gmaps.place(place_id, fields=CAMPOS)["result"]

    datos.append({
        "place_id": place_id,
        "nombre": detalles.get("name", ""),
        "rating": detalles.get("rating"),
        "total_resenas": detalles.get("user_ratings_total", 0),
        "direccion": detalles.get("formatted_address", ""),
        "lat": detalles.get("geometry", {}).get("location", {}).get("lat"),
        "lng": detalles.get("geometry", {}).get("location", {}).get("lng"),
        "nivel_precio": detalles.get("price_level"),
        "abierto_ahora": detalles.get("opening_hours", {}).get("open_now"),
        "tipos": ", ".join(detalles.get("types", []))
    })

df = pd.DataFrame(datos)
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()

## Guardado en CSV

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ruta_salida = f"../data/competidores_{timestamp}.csv"
df.to_csv(ruta_salida, index=False)
print(f"Guardado en: {ruta_salida}")

## Análisis exploratorio inicial

In [ ]:
df[["rating", "total_resenas", "nivel_precio"]].describe().round(2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["rating"].dropna().hist(bins=10, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Distribución de Ratings")
axes[0].set_xlabel("Rating")

df["total_resenas"].dropna().hist(bins=20, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Cantidad de Reseñas por Local")
axes[1].set_xlabel("Nº de Reseñas")

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 por rating (mínimo 20 reseñas para filtrar ruido)
df[df["total_resenas"] >= 20].sort_values("rating", ascending=False).head(10)[
    ["nombre", "rating", "total_resenas", "nivel_precio", "direccion"]
]